## Feature Engineering Summary

In [1]:
import pandas as pd

df = pd.read_csv("cleaned_data.csv")

### Tenure Segmentation

Customers are grouped by how long they stayed with the company.
This helps identify churn patterns across customer lifecycle stages.

In [2]:
df["tenure_group"] = pd.cut(
    df["Tenure Months"],
    bins=[0, 12, 24, 48, 72],
    labels=["0-1 year", "1-2 years", "2-4 years", "4-6 years"],
    include_lowest=True
)

In [3]:
df["tenure_group"].value_counts()

tenure_group
4-6 years    2239
0-1 year     2133
2-4 years    1594
1-2 years    1024
Name: count, dtype: int64

### Service Count

Measures how many services a customer uses.
Customers with more services are usually more engaged and less likely to churn.

In [4]:
services = [
    "Phone Service",
    "Multiple Lines",
    "Internet Service",
    "Online Security",
    "Online Backup",
    "Device Protection",
    "Tech Support",
    "Streaming TV",
    "Streaming Movies"
]

df["service_count"] = df[services].apply(lambda x: (x == 1).sum(), axis=1)

In [5]:
df["service_count"].value_counts()

service_count
1    6311
0     679
Name: count, dtype: int64

### Average Monthly Value

Calculated as:
Total Charges / Tenure

This represents the customer's historical value rather than current billing.

In [6]:
df["avg_monthly_value"] = df["Total Charges"] / (df["Tenure Months"] + 1)

### Value Change

Difference between current monthly charges and historical average.

Insight:
- Positive: customer paying more now
- Negative: customer value decreasing (possible churn signal)

In [7]:
df["value_change"] = df["Monthly Charges"] - df["avg_monthly_value"]

In [8]:
df["revenue_per_month"] = df["Monthly Charges"]

### Engagement Score

A weighted metric combining:
- Service usage (50%)
- Tenure (30%)
- Monthly charges (20%)

This captures overall customer engagement.

Note:
Weights are heuristic and can be tuned using model performance.

In [9]:
df["engagement_score"] = (
    df["service_count"] * 0.5 +
    df["Tenure Months"] * 0.3 +
    df["Monthly Charges"] * 0.2
)

In [10]:
df["is_new_customer"] = (df["Tenure Months"] <= 12).astype(int)

### Contract Risk

Customers on flexible contracts (month-to-month) are more likely to churn,
so higher risk values are assigned.

In [11]:
df["Contract"] = df["Contract"].str.strip().str.lower()

df["contract_risk"] = df["Contract"].map({
    "month-to-month": 2,
    "one year": 1,
    "two year": 0
})

In [12]:
print(df["Contract"].unique())
print(df["contract_risk"].unique())

['month-to-month' 'two year' 'one year']
[2 0 1]


### Payment Risk

Customers using electronic payment methods may have different churn behavior.

(Binary encoding applied)

In [13]:
df["payment_risk"] = df["Payment Method"].apply(
    lambda x: 1 if "Electronic" in x else 0
)

In [14]:
df["monthly_x_tenure"] = df["Monthly Charges"] * df["Tenure Months"]

### Behavioral Flags

Binary features capturing key customer characteristics:
- Internet usage
- Family status
- Engagement level
- High-risk profile

These simplify model learning and improve interpretability.

In [15]:
df["has_internet"] = (df["Internet Service"] != "No").astype(int)

In [16]:
df["has_family"] = (
    (df["Partner"] == "Yes") |
    (df["Dependents"] == "Yes")
).astype(int)

In [17]:
df["low_engagement"] = (df["service_count"] <= 2).astype(int)

In [18]:
df["high_risk"] = (
    (df["Contract"] == "month-to-month") &
    (df["Tenure Months"] < 12)
).astype(int)

### Dropping Raw Tenure

The original tenure column is removed since its information
is already captured in engineered features like:
- tenure_group
- engagement_score

In [19]:
df.drop(["Tenure Months"], axis=1, inplace=True)

In [20]:
df.head()

,Senior Citizen,Partner,Dependents,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,...,revenue_per_month,engagement_score,is_new_customer,contract_risk,payment_risk,monthly_x_tenure,has_internet,has_family,low_engagement,high_risk
0,0,0,0,1,No,DSL,Yes,Yes,No,No,...,53.85,11.87,1,2,0,107.7,1,0,1,1
1,0,0,1,1,No,Fiber optic,No,No,No,No,...,70.70,15.24,1,2,1,141.4,1,0,1,1
2,0,0,1,1,Yes,Fiber optic,No,No,Yes,No,...,99.65,22.83,1,2,1,797.2,1,0,1,1
3,0,1,1,1,Yes,Fiber optic,No,No,Yes,Yes,...,104.80,29.86,0,2,1,2934.4,1,0,1,0
4,0,0,1,1,Yes,Fiber optic,No,Yes,Yes,No,...,103.70,35.94,0,2,0,5081.3,1,0,1,0


In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6990 entries, 0 to 6989
Data columns (total 42 columns):
 #   Column                                    Non-Null Count  Dtype   
---  ------                                    --------------  -----   
 0   Senior Citizen                            6990 non-null   int64   
 1   Partner                                   6990 non-null   int64   
 2   Dependents                                6990 non-null   int64   
 3   Phone Service                             6990 non-null   int64   
 4   Multiple Lines                            6990 non-null   object  
 5   Internet Service                          6990 non-null   object  
 6   Online Security                           6990 non-null   object  
 7   Online Backup                             6990 non-null   object  
 8   Device Protection                         6990 non-null   object  
 9   Tech Support                              6990 non-null   object  
 10  Streaming TV            

In [22]:
df.describe()
df.head()

,Senior Citizen,Partner,Dependents,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,...,revenue_per_month,engagement_score,is_new_customer,contract_risk,payment_risk,monthly_x_tenure,has_internet,has_family,low_engagement,high_risk
0,0,0,0,1,No,DSL,Yes,Yes,No,No,...,53.85,11.87,1,2,0,107.7,1,0,1,1
1,0,0,1,1,No,Fiber optic,No,No,No,No,...,70.70,15.24,1,2,1,141.4,1,0,1,1
2,0,0,1,1,Yes,Fiber optic,No,No,Yes,No,...,99.65,22.83,1,2,1,797.2,1,0,1,1
3,0,1,1,1,Yes,Fiber optic,No,No,Yes,Yes,...,104.80,29.86,0,2,1,2934.4,1,0,1,0
4,0,0,1,1,Yes,Fiber optic,No,Yes,Yes,No,...,103.70,35.94,0,2,0,5081.3,1,0,1,0


In [23]:
df.to_csv("engineered_data.csv", index=False)

## Feature Engineering Summary & Impact


To improve churn prediction, I created features that capture:

- Customer longevity --> tenure_group
- Service usage --> service_count
- Customer value --> avg_monthly_value
- Behavioral engagement --> engagement_score
- Risk indicators --> contract_risk, payment_risk
- Customer lifecycle --> is_new_customer


The engineered features transform raw customer data into meaningful signals:

- Behavioral features: engagement_score, service_count
- Risk features: contract_risk, payment_risk, high_risk
- Value features: avg_monthly_value, value_change

These features are expected to:
- Improve churn prediction accuracy
- Enable better customer segmentation
- Support decision-making systems (retention strategies)

## Feature Risks & Considerations

- monthly_x_tenure may cause data leakage
- engagement_score weights are heuristic
- avg_monthly_value depends on tenure quality

These will be validated during modeling.


For the next step I'm going to train machine learning models using these features, but before that I will segment the customers